# 11 - Overlay: ticker mentions vs price

Ticker daily **mention count** (`daily_ticker_counts`) vs its **Bloomberg
close** on a shared date axis, clipped to the window set in `update_data.py`.
No analysis - just the two lines. Edit `TICKER` and re-run.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

# find project root whether run from notebooks/ or the project root
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# THE WINDOW comes from update_data.py - the SAME two dates that control live vs
# backtest for the whole pipeline. END_DATE = '' means live (up to the newest
# data); set END_DATE to a date in update_data.py to freeze a past regime.
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    """Keep only the dates inside the window (s indexed by date)."""
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    """Keep only the rows whose date column is inside the window."""
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError(
            'prices.parquet not found - run  python pull_bloomberg_prices.py  '
            'on the work laptop first (needs the Bloomberg Terminal).')
    px = pd.read_parquet(PRICES_PATH)
    px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    one = prices[prices['symbol'] == symbol].sort_values('date')
    if one.empty:
        print('WARNING: no price rows for', symbol, '- is it in the pull universe?')
    return clip_series(one.set_index('date')['px_last'])


In [ ]:
# ============ EDIT THESE ============
TICKER = 'GME'      # which ticker to overlay
ROLL   = 7          # smoothing for mentions in days (1 = raw daily counts)
# ====================================

In [ ]:
counts = pd.read_parquet(os.path.join(P, 'daily_ticker_counts.parquet'))
counts['date'] = pd.to_datetime(counts['date'])

m = (counts[counts['ticker'] == TICKER]
     .sort_values('date').set_index('date')['mention_count'])
m = m.asfreq('D').fillna(0)
if ROLL > 1:
    m = m.rolling(ROLL).mean()
m = clip_series(m)                       # focus the window from update_data.py

prices = load_prices()
px = price_series(prices, TICKER)

fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.plot(m.index, m.values, color='tab:blue', linewidth=1.6,
         label=f'{TICKER} mentions ({ROLL}d)')
ax1.set_ylabel(f'mentions per day ({ROLL}d avg)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(px.index, px.values, color='tab:red', linewidth=1.6,
         label=f'{TICKER} price (PX_LAST)')
ax2.set_ylabel('price (USD)', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

ax1.set_title(f'{TICKER}: retail mentions vs price')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()